In [ ]:
# evaluate_viverse.py
import torch
import os
import time
import gc  # Thêm thư viện dọn dẹp bộ nhớ
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessorList
from bert_score import BERTScorer

# IMPORT CLASS TỪ FILE VỪA TẠO
from lucbat_processor import StrictLucBatProcessor

# ==========================================
# 1. HÀM CHẤM ĐIỂM LUẬT THƠ 
# ==========================================
TRAC_CHARS = set("áắấéếíóốớúứýảẳẩẻểỉỏổởủửỷãẵẫẽễĩõỗỡũữỹạặậẹệịọộợụựỵ")

def get_tone(word):
    word = word.lower()
    has_alpha = any(c.isalpha() for c in word)
    if not has_alpha: return None
    return "T" if any(c in TRAC_CHARS for c in word) else "B"

def remove_tones(word):
    s1 = u'ÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝàáâãèéêìíòóôõùúýĂăĐđĨĩŨũƠơƯưẠạẢảẤấẦầẨẩẪẫẬậẮắẰằẲẳẴẵẶặẸẹẺẻẼẽẾếỀềỂểỄễỆệỈỉỊịỌọỎỏỐốỒồỔổỖỗỘộỚớỜờỞởỠỡỢợỤụỦủỨứỪừỬửỮữỰựỲỳỴỵỶỷỸỹ'
    s0 = u'AAAAEEEIIOOOOUUYaaaaeeeiioooouuyAaDdIiUuOoUuAaAaAaAaAaAaAaAaAaAaAaAaEeEeEeEeEeEeEeEeIiIiOoOoOoOoOoOoOoOoOoOoOoOoUuUuUuUuUuUuUuYyYyYyYy'
    s = ''
    for c in word:
        if c in s1: s += s0[s1.index(c)]
        else: s += c
    return s

def get_rhyme_part(word):
    word = remove_tones(word.lower().strip())
    consonants = ['ngh', 'ch', 'gh', 'gi', 'kh', 'ng', 'nh', 'ph', 'qu', 'th', 'tr', 
                  'b', 'c', 'd', 'đ', 'g', 'h', 'k', 'l', 'm', 'n', 'p', 'q', 'r', 's', 't', 'v', 'x']
    for c in consonants:
        if word.startswith(c): return word[len(c):]
    return word 

def is_rhyme(word1, word2):
    if not word1 or not word2: return False
    return get_rhyme_part(word1) == get_rhyme_part(word2)

def evaluate_luc_bat_rules(cau_luc, cau_bat_gen):
    luc_words = cau_luc.strip().split()
    bat_words = cau_bat_gen.strip().split()
    
    score_length = 0.0
    score_tone = 0.0
    score_rhyme = 0.0
    
    if len(bat_words) == 8: score_length = 10.0
        
    if len(bat_words) >= 8:
        tone_score_per_word = 30.0 / 4.0
        if get_tone(bat_words[1]) == "B": score_tone += tone_score_per_word
        if get_tone(bat_words[3]) == "T": score_tone += tone_score_per_word
        if get_tone(bat_words[5]) == "B": score_tone += tone_score_per_word
        if get_tone(bat_words[7]) == "B": score_tone += tone_score_per_word
        
    if len(luc_words) >= 6 and len(bat_words) >= 6:
        if is_rhyme(luc_words[5], bat_words[5]): score_rhyme = 60.0
            
    return score_length + score_tone + score_rhyme, score_length, score_tone, score_rhyme

# ==========================================
# 2. CHẠY KIỂM THỬ VÀ ĐÁNH GIÁ TRÊN TOÀN BỘ TẬP DỮ LIỆU
# ==========================================
DATASET_FILE = "test.txt"
test_data = []
with open(DATASET_FILE, 'r', encoding='utf-8') as f:
    lines = [line.strip() for line in f if line.strip()]

i = 0
while i < len(lines) - 1:
    if len(lines[i].split()) == 6 and len(lines[i+1].split()) == 8:
        test_data.append((lines[i], lines[i+1]))
        i += 2
    else: i += 1

# Đã gỡ bỏ giới hạn 10% ở đây để chạy trên 100% dữ liệu

print("Đang tải Tokenizer và Mô hình Sinh thơ...")
model_path = "./qwen-lucbat-model" 
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True)
model.eval()

# GỌI CLASS ÉP LUẬT TỪ FILE NGOÀI (Chỉ ép 2 dòng vì ta đang test sinh câu Bát từ câu Lục)
processor = StrictLucBatProcessor(tokenizer, max_total_lines=2)
logits_processor_list = LogitsProcessorList([processor])

print("Đang khởi tạo BERTScorer (PhoBERT)...")
scorer = BERTScorer(model_type="vinai/phobert-base", num_layers=9, rescale_with_baseline=False)

print("\n" + "="*60)
print(f"BẮT ĐẦU ĐÁNH GIÁ (CÓ BỘ ÉP LUẬT TỔNG HỢP) - TẤT CẢ {len(test_data)} MẪU (TOÀN BỘ DỮ LIỆU)")
print("="*60)

total_rule = 0; total_len = 0; total_tone = 0; total_rhyme = 0
generated_bats = []
reference_bats = []

start_time = time.time()

for i, (cau_luc, cau_bat_ref) in enumerate(test_data):
    prompt = cau_luc + "\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=25, 
            logits_processor=logits_processor_list, 
            do_sample=True,          
            temperature=0.8,        
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    cau_bat_gen = full_text.replace(cau_luc, "").strip().split("\n")[0].strip()
    
    rule_score, len_s, tone_s, rhyme_s = evaluate_luc_bat_rules(cau_luc, cau_bat_gen)
    
    total_rule += rule_score; total_len += len_s; total_tone += tone_s; total_rhyme += rhyme_s
    generated_bats.append(cau_bat_gen)
    reference_bats.append(cau_bat_ref)
    
    if (i + 1) % 10 == 0 or (i + 1) == len(test_data):
        print(f"Đã xử lý [{i+1}/{len(test_data)}] mẫu...")

# ==========================================
# CẬP NHẬT: TÍNH PHO-BERT SCORE THEO TỪNG BATCH ĐỂ TRÁNH TRÀN RAM/VRAM
# ==========================================
print("\nĐang tính toán PhoBERTScore cho toàn bộ tập test (xử lý theo batch để tránh tràn RAM)...")

batch_size = 64 # Bạn có thể giảm xuống 32 hoặc 16 nếu vẫn bị tràn RAM GPU
all_f1_scores = []

with torch.no_grad():
    for idx in range(0, len(generated_bats), batch_size):
        batch_gen = generated_bats[idx : idx + batch_size]
        batch_ref = reference_bats[idx : idx + batch_size]
        
        # Tính điểm cho batch hiện tại
        P, R, F1 = scorer.score(batch_gen, batch_ref)
        
        # Chuyển Tensor sang list float ngay lập tức để ngắt liên kết đồ thị tính toán
        all_f1_scores.extend(F1.cpu().tolist())
        
        # Dọn rác
        del P, R, F1, batch_gen, batch_ref
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        
        # In tiến trình
        if (idx // batch_size + 1) % 50 == 0:
            current_processed = min(idx + batch_size, len(generated_bats))
            print(f"  Đã tính PhoBERTScore: [{current_processed}/{len(generated_bats)}] mẫu...")

# Tính trung bình điểm số từ list float
avg_phobert_f1 = (sum(all_f1_scores) / len(all_f1_scores)) * 100 

n = len(test_data)
eval_time = time.time() - start_time

print("\n" + "="*60)
print("BÁO CÁO ĐÁNH GIÁ MÔ HÌNH QWEN 3.5 0.8B (TOÀN BỘ DỮ LIỆU)")
print("="*60)
print(f"Tổng số mẫu test:      {n}")
print(f"Thời gian chạy:        {eval_time:.1f} giây (~{eval_time/60:.1f} phút / ~{eval_time/3600:.2f} giờ)")
print(f"1. ĐIỂM LUẬT THƠ (Rule-based): {total_rule/n:.1f} / 100")
print(f"   - Độ dài (10%):     {total_len/n:.1f}%")
print(f"   - Thanh điệu (30%): {total_tone/n:.1f}%")
print(f"   - Gieo vần (60%):   {total_rhyme/n:.1f}%")
print(f"2. PHO-BERT SCORE (Semantic) : {avg_phobert_f1:.1f}%")
print("="*60)

Đã xử lý [10700/281181] mẫu...
Đã xử lý [10710/281181] mẫu...
Đã xử lý [10720/281181] mẫu...
Đã xử lý [10730/281181] mẫu...
Đã xử lý [10740/281181] mẫu...
Đã xử lý [10750/281181] mẫu...
Đã xử lý [10760/281181] mẫu...
Đã xử lý [10770/281181] mẫu...
Đã xử lý [10780/281181] mẫu...
Đã xử lý [10790/281181] mẫu...
Đã xử lý [10800/281181] mẫu...
Đã xử lý [10810/281181] mẫu...
Đã xử lý [10820/281181] mẫu...
Đã xử lý [10830/281181] mẫu...
Đã xử lý [10840/281181] mẫu...
Đã xử lý [10850/281181] mẫu...
Đã xử lý [10860/281181] mẫu...
Đã xử lý [10870/281181] mẫu...
Đã xử lý [10880/281181] mẫu...
Đã xử lý [10890/281181] mẫu...
Đã xử lý [10900/281181] mẫu...
Đã xử lý [10910/281181] mẫu...
Đã xử lý [10920/281181] mẫu...
Đã xử lý [10930/281181] mẫu...
Đã xử lý [10940/281181] mẫu...
Đã xử lý [10950/281181] mẫu...
Đã xử lý [10960/281181] mẫu...
Đã xử lý [10970/281181] mẫu...
Đã xử lý [10980/281181] mẫu...
Đã xử lý [10990/281181] mẫu...
Đã xử lý [11000/281181] mẫu...
Đã xử lý [11010/281181] mẫu...
Đã xử lý

: 